library: gitsource, which downloads files from a GitHub repository. we will pull the lesson pages straight from the course repository. We will use the commit 8c1834d to make sure everyone works with the exact same data.

We will use gitsource for that:

In [1]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

In [2]:
documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)

**Q1. How many lesson pages**

In [4]:
len(documents)

72

**Q2. Indexing and searching**

Index the documents with minsearch - make content a text field and filename a keyword field. Then search with this query:

How does the agentic loop keep calling the model until it stops?

What's the filename of the first result?

In [3]:
from minsearch import Index

index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

index.fit(documents)

In [ ]:
question = "How does the agentic loop keep calling the model until it stops?"

search_results = index.search(
    question
)

# search_results

In [3]:
documents[0]

{'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phone uses a simp

**Q3. RAG**

Build a RAG over the index from Q2 and answer the query:

How does the agentic loop keep calling the model until it stops?

Use gpt-5.4-mini. How many input (prompt) tokens did we send to the model for this request?

In [5]:
from dotenv import load_dotenv
load_dotenv()

from rag_helper_hw1 import DocumentRAG
from openai import OpenAI

openai_client = OpenAI()

assistant = DocumentRAG(index=index, llm_client=openai_client)

Option 1 — modify llm() to return usage too

In [8]:
answer, usage = assistant.rag(question)
print(answer)
print(f"Input tokens: {usage.input_tokens}")

It keeps calling the model inside a `while True` loop.  

Each iteration:
1. Sends the full message history to the model.
2. Checks the response for any `function_call` items.
3. Runs those tools and appends the results to `messages`.
4. If there were no function calls this turn, it `break`s.

So the stop condition is simply: **no function calls in the model’s response**.
Input tokens: 7126


Option 2 — call build_prompt directly, skip rag()

In [12]:
# Build prompt manually
search_results = assistant.search(question)
prompt = assistant.build_prompt(question, search_results)

# Call API directly
response = openai_client.responses.create(
    model='gpt-5.4-mini',
    input=[
        {'role': 'developer', 'content': assistant.instructions},
        {'role': 'user', 'content': prompt}
    ]
)

print(response.output_text)
print(f"Input tokens: {response.usage.input_tokens}")

It keeps calling the model inside a `while True` loop. After each model response, the code checks whether there were any `function_call` items.

- If there was a function call, it runs the tool, appends the tool result to `messages`, and loops again.
- If there were no function calls, it breaks out of the loop.

So the stop condition is:

```python
if has_function_calls == False:
    break
```

In other words, it keeps going until the model returns a final message with no more tool calls.
Input tokens: 7126


**Q4. Chunking**

How many chunks do you get?

In [5]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

len(chunks)

295

**Q5. RAG with chunking**

Compare the input tokens with Q3. How many fewer input tokens does the chunked version send?

In [6]:
from minsearch import Index

index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

index.fit(chunks)

In [7]:
from dotenv import load_dotenv
load_dotenv()

from rag_helper_hw1 import DocumentRAG
from openai import OpenAI

openai_client = OpenAI()

assistant = DocumentRAG(index=index, llm_client=openai_client)

In [8]:
question = "How does the agentic loop keep calling the model until it stops?"

answer, usage = assistant.rag(question)
print(answer)
print(f"Input tokens: {usage.input_tokens}")

It keeps looping with a `while True` loop and checks each model response for `function_call` items.

- If the response contains a function call, the code runs the tool, appends the tool result to the message history, and sets `has_function_calls = True`.
- If the response contains only a final `message` and no function calls, `has_function_calls` stays `False`.
- At the end of the iteration, `if has_function_calls == False: break` stops the loop.

So the loop stops when the model returns a response with no more tool calls.
Input tokens: 2309


**Q6. Turning it into an agent**

In [2]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

In [3]:
documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)

In [4]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)



In [5]:
from minsearch import Index

index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

index.fit(chunks)

In [ ]:
index.docs

In [14]:
def search(query: str) -> list[dict]:
    """
    Search the document index for chunks matching the given query.
    """
    return index.search(
        query,
        num_results=5,
        boost_dict={"content": 1.0},
        filter_dict={}
    )

In [8]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

In [15]:
agent_tools = Tools()
agent_tools.add_tool(search)
agent_tools.get_tools()

[{'type': 'function',
  'name': 'search',
  'description': 'Search the document index for chunks matching the given query.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'query parameter'}},
   'required': ['query'],
   'additionalProperties': False}}]

In [16]:
INSTRUCTIONS = '''
You're a course teaching assistant. Answer the student's question using the search tool. Make multiple searches with different keywords before answering.
'''

chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=INSTRUCTIONS,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model="gpt-5.4-mini")
)

In [17]:
result = runner.loop(
    prompt="How does the agentic loop work, and how is it different from plain RAG?",
    callback=callback,
)

result.cost
result.all_messages

-> Response received


-> Response received


[EasyInputMessage(content="\nYou're a course teaching assistant. Answer the student's question using the search tool. Make multiple searches with different keywords before answering.\n", role='developer', phase=None, type=None),
 EasyInputMessage(content='How does the agentic loop work, and how is it different from plain RAG?', role='user', phase=None, type=None),
 ResponseFunctionToolCall(arguments='{"query":"agentic loop RAG difference retrieval generation iteration"}', call_id='call_xWk9rr4wcn10SCzL6wLX18VO', name='search', type='function_call', id='fc_01bd60a020e152cc006a38f2abd06081a3aa744679eef5081e', namespace=None, status='completed'),
 ResponseFunctionToolCall(arguments='{"query":"agentic loop planner act observe revise retrieve"}', call_id='call_EdEGekxJMiPS6AFG0MfdZVv6', name='search', type='function_call', id='fc_01bd60a020e152cc006a38f2abd07481a391c64bf14e934289', namespace=None, status='completed'),
 ResponseFunctionToolCall(arguments='{"query":"plain RAG retrieve augment